# Phase 1C re-run — Iloilo only

Iloilo had one transmission substation in OSM (`sub_osm_4`), so Phase 1C generated only 24 distribution buses for it (36 MW peak load). After Steps 1–3 of the v1 integration, Iloilo now has **7 substation-class buses** (1 OSM + 6 v1-curated). This notebook regenerates Iloilo's distribution layer using all 7 as roots, with the exact same parameters Phase 1C used elsewhere — only the inputs changed.

**Scope:** Iloilo only. All other provinces left untouched (no risk of drift on Cebu / Negros / Leyte / Bohol etc.).

**Pipeline:**
1. Drop existing Iloilo distribution buses (those with `province='Iloilo' AND bus_type='distribution'`) and their incident lines.
2. For each Iloilo substation-class root (`bus_type ∈ {substation, substation_synth, generator, hvdc, bess}`), run 4 feeders × 6 branches with the Phase 1C generator.
3. Append new buses and lines.

**Expected output:** 7 × 24 = 168 new distribution buses + 168 new lines for Iloilo, raising its synthetic peak load from 36 MW to **252 MW** — far closer to the published ~250 MW real Iloilo peak demand.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

PROC_DIR  = Path('../backend/data/processed')
BOUND_DIR = Path('../backend/data/boundaries')
WGS = 'EPSG:4326'
UTM = 'EPSG:32651'

DIST_VOLTAGE_KV     = 13.8
N_FEEDERS_PER_ROOT  = 4
BRANCHES_PER_FEEDER = 6
LOAD_PER_BUS_MW     = 1.5
PF                  = 0.9
TAN_PHI             = math.tan(math.acos(PF))

DIST_R_OHM_PER_KM = 0.10
DIST_X_OHM_PER_KM = 0.15
DIST_MAX_I_KA     = 0.30

FEEDER_MIN_KM    = 5.0
FEEDER_MAX_KM    = 25.0
MAX_REJECT_TRIES = 200

TARGET_PROVINCE = 'Iloilo'

# Province-specific seed (different from Phase 1C's 20260511 to avoid bus_id collisions on regen)
RNG = np.random.default_rng(seed=20260512)

In [2]:
buses = pd.read_csv(PROC_DIR / 'buses.csv')
lines = pd.read_csv(PROC_DIR / 'lines.csv')
provinces = gpd.read_file(BOUND_DIR / 'psgc_provinces.geojson')

print(f'Inputs: {len(buses)} buses, {len(lines)} lines')
iloilo_poly = provinces[provinces['province'] == TARGET_PROVINCE].to_crs(UTM).iloc[0].geometry
print(f'Iloilo province polygon loaded (area = {iloilo_poly.area/1e6:.0f} km²)')

Inputs: 2396 buses, 2409 lines
Iloilo province polygon loaded (area = 4711 km²)


## §1 — Identify Iloilo roots and existing distribution

In [3]:
ROOT_TYPES = ['substation', 'substation_synth', 'generator', 'hvdc', 'bess']
iloilo_roots = buses[
    (buses['province'] == TARGET_PROVINCE) &
    (buses['bus_type'].isin(ROOT_TYPES))
].copy()
print(f'Iloilo roots ({len(iloilo_roots)}):')
print(iloilo_roots[['bus_id', 'name', 'voltage_kv', 'bus_type', 'data_source']].to_string(index=False))

iloilo_dist = buses[
    (buses['province'] == TARGET_PROVINCE) &
    (buses['bus_type'] == 'distribution')
].copy()
print(f'\nExisting Iloilo distribution buses (will be dropped): {len(iloilo_dist)}')
print(f'Their bus_ids start with: {sorted(iloilo_dist["bus_id"].str[:20].unique().tolist())[:5]} ...')

Iloilo roots (7):
       bus_id                                  name  voltage_kv   bus_type data_source
    sub_osm_4         Concepcion substation, Iloilo       138.0 substation         osm
  v1_08bantap             Bantap substation, Iloilo        69.0 substation  v1_curated
 v1_08barotac            Barotac substation, Iloilo       138.0 substation  v1_curated
  v1_08dingle             Dingle substation, Iloilo       138.0 substation  v1_curated
 v1_08iloilo1 Iloilo 1 main substation, Iloilo City       138.0 substation  v1_curated
  v1_08snjose           San Jose substation, Iloilo       138.0 substation  v1_curated
v1_08stbarbra      Santa Barbara substation, Iloilo       138.0 substation  v1_curated

Existing Iloilo distribution buses (will be dropped): 24
Their bus_ids start with: ['sub_osm_4_F0_B0', 'sub_osm_4_F0_B1', 'sub_osm_4_F0_B2', 'sub_osm_4_F0_B3', 'sub_osm_4_F0_B4'] ...


In [4]:
# Drop them: buses where bus_id is in iloilo_dist, and lines incident on any of those bus_ids
drop_bus_ids = set(iloilo_dist['bus_id'])
buses_kept = buses[~buses['bus_id'].isin(drop_bus_ids)].copy()
lines_kept = lines[
    ~lines['from_bus'].isin(drop_bus_ids) & ~lines['to_bus'].isin(drop_bus_ids)
].copy()

print(f'After drop: {len(buses_kept)} buses ({len(buses)} - {len(drop_bus_ids)}), '
      f'{len(lines_kept)} lines ({len(lines)} - {len(lines)-len(lines_kept)})')

After drop: 2372 buses (2396 - 24), 2385 lines (2409 - 24)


## §2 — Phase 1C feeder generator (verbatim from notebook 03)

In [5]:
def random_point_in_polygon(poly, around_point_m, min_dist_m, max_dist_m, bearing_rad, rng):
    wedge_half_width = math.radians(35)
    for _ in range(MAX_REJECT_TRIES):
        d = rng.uniform(min_dist_m, max_dist_m)
        theta = rng.uniform(bearing_rad - wedge_half_width, bearing_rad + wedge_half_width)
        x = around_point_m.x + d * math.cos(theta)
        y = around_point_m.y + d * math.sin(theta)
        p = Point(x, y)
        if poly.contains(p):
            return p
    return None

def synthesize_feeders_for_root(root_row, province_poly_m, n_feeders, branches, rng):
    root_pt_m = gpd.GeoSeries(
        [Point(root_row['lon'], root_row['lat'])], crs=WGS
    ).to_crs(UTM).iloc[0]
    bus_rows, line_rows = [], []
    bearings = np.linspace(0, 2 * math.pi, n_feeders, endpoint=False)
    bearings += rng.uniform(0, 2 * math.pi / n_feeders)
    for f_idx, bearing in enumerate(bearings):
        prev_bus_id = root_row['bus_id']
        prev_pt = root_pt_m
        for b_idx in range(branches):
            min_d = FEEDER_MIN_KM * 1000 * (b_idx + 1) / branches
            max_d = FEEDER_MAX_KM * 1000 * (b_idx + 1) / branches
            pt = random_point_in_polygon(province_poly_m, root_pt_m, min_d, max_d, bearing, rng)
            if pt is None:
                break
            new_bus_id = f'{root_row["bus_id"]}_F{f_idx}_B{b_idx}'
            bus_rows.append({
                '_pt_m': pt,
                'bus_id': new_bus_id, 'name': new_bus_id,
                'voltage_kv': DIST_VOLTAGE_KV,
                'province': root_row['province'], 'island': root_row['island'],
                'bus_type': 'distribution',
                'p_mw': LOAD_PER_BUS_MW, 'q_mvar': LOAD_PER_BUS_MW * TAN_PHI,
                'is_synthetic': True, 'data_source': 'synthetic',
            })
            line_rows.append({
                'from_bus': prev_bus_id, 'to_bus': new_bus_id,
                'voltage_kv': DIST_VOLTAGE_KV,
                'length_km':  prev_pt.distance(pt) / 1000,
                'r_ohm_per_km': DIST_R_OHM_PER_KM, 'x_ohm_per_km': DIST_X_OHM_PER_KM,
                'max_i_ka': DIST_MAX_I_KA,
                'is_submarine': False, 'cable_type': 'overhead',
                'is_synthetic': True, 'data_source': 'synthetic',
            })
            prev_bus_id = new_bus_id
            prev_pt = pt
    return bus_rows, line_rows

## §3 — Run synthesis for each Iloilo root

In [6]:
all_bus_rows, all_line_rows = [], []
for _, root in iloilo_roots.iterrows():
    b_rows, l_rows = synthesize_feeders_for_root(
        root, iloilo_poly, N_FEEDERS_PER_ROOT, BRANCHES_PER_FEEDER, RNG)
    print(f'  {root["bus_id"]:<18} → {len(b_rows):3d} buses, {len(l_rows):3d} lines')
    all_bus_rows.extend(b_rows)
    all_line_rows.extend(l_rows)

print(f'\nTotal new Iloilo distribution: {len(all_bus_rows)} buses, {len(all_line_rows)} lines')
print(f'Total new peak load: {len(all_bus_rows) * LOAD_PER_BUS_MW:.1f} MW')

  sub_osm_4          →  24 buses,  24 lines
  v1_08bantap        →  20 buses,  20 lines
  v1_08barotac       →  18 buses,  18 lines
  v1_08dingle        →  24 buses,  24 lines
  v1_08iloilo1       →  16 buses,  16 lines
  v1_08snjose        →  24 buses,  24 lines
  v1_08stbarbra      →  24 buses,  24 lines

Total new Iloilo distribution: 150 buses, 150 lines
Total new peak load: 225.0 MW


## §4 — Re-project, finalize schema, append

In [7]:
new_buses_m = gpd.GeoDataFrame(
    all_bus_rows, geometry=[r['_pt_m'] for r in all_bus_rows], crs=UTM
).to_crs(WGS)
new_buses_df = pd.DataFrame({
    'bus_id':       new_buses_m['bus_id'],
    'name':         new_buses_m['name'],
    'lat':          new_buses_m.geometry.y,
    'lon':          new_buses_m.geometry.x,
    'voltage_kv':   new_buses_m['voltage_kv'],
    'province':     new_buses_m['province'],
    'island':       new_buses_m['island'],
    'bus_type':     new_buses_m['bus_type'],
    'p_mw':         new_buses_m['p_mw'],
    'q_mvar':       new_buses_m['q_mvar'],
    'is_synthetic': new_buses_m['is_synthetic'],
    'data_source':  new_buses_m['data_source'],
})

# Renumber line_ids starting from the highest existing line_synth_
next_idx = int(lines_kept['line_id'].str.extract(r'line_synth_(\d+)').astype(float).max().iloc[0] or 0) + 1
new_lines_df = pd.DataFrame(all_line_rows)
new_lines_df['line_id'] = [f'line_synth_{i:05d}' for i in range(next_idx, next_idx + len(new_lines_df))]

# Align to schemas of the kept tables
buses_out = pd.concat([buses_kept, new_buses_df.reindex(columns=buses_kept.columns, fill_value=np.nan)], ignore_index=True)
lines_out = pd.concat([lines_kept, new_lines_df.reindex(columns=lines_kept.columns, fill_value=np.nan)], ignore_index=True)

# Validate
assert buses_out['bus_id'].is_unique
assert lines_out['line_id'].is_unique
valid = set(buses_out['bus_id'])
assert lines_out['from_bus'].isin(valid).all() and lines_out['to_bus'].isin(valid).all()

buses_out.to_csv(PROC_DIR / 'buses.csv', index=False)
lines_out.to_csv(PROC_DIR / 'lines.csv', index=False)
print(f'Wrote buses.csv ({len(buses_out)} rows)')
print(f'Wrote lines.csv ({len(lines_out)} rows)')

Wrote buses.csv (2522 rows)
Wrote lines.csv (2535 rows)


## §5 — Verify Iloilo coverage and load

In [8]:
iloilo_after = buses_out[buses_out['province'] == TARGET_PROVINCE]
print('Iloilo buses by bus_type after re-run:')
print(iloilo_after['bus_type'].value_counts().to_string())

iloilo_load = iloilo_after['p_mw'].sum()
print(f'\nIloilo peak load: {iloilo_load:.1f} MW')
print(f'  Was 36 MW after Phase 1C (one root).')
print(f'  Target ~250 MW (published Iloilo peak demand).')

Iloilo buses by bus_type after re-run:
bus_type
distribution    150
substation        7
tower             1

Iloilo peak load: 225.0 MW
  Was 36 MW after Phase 1C (one root).
  Target ~250 MW (published Iloilo peak demand).


In [9]:
# Connectivity: Iloilo now should have one big connected component (all roots + their feeders + the Panay MST backbone)
import networkx as nx
G = nx.Graph()
G.add_nodes_from(buses_out['bus_id'])
G.add_edges_from(zip(lines_out['from_bus'], lines_out['to_bus']))

iloilo_bus_set = set(iloilo_after['bus_id'])
subg = G.subgraph(iloilo_bus_set)
print(f'Iloilo component count (incl. distribution): {nx.number_connected_components(subg)}')
sizes = sorted([len(c) for c in nx.connected_components(subg)], reverse=True)
print(f'Top sizes: {sizes[:5]}')

# Whole grid component count
ncc_all = nx.number_connected_components(G)
print(f'\nWhole-grid component count: {ncc_all}')

Iloilo component count (incl. distribution): 1
Top sizes: [158]

Whole-grid component count: 66
